# Test ba detector trên Tiny-GenImage

Notebook này chạy inference-only cho cùng một Tiny-GenImage validation cohort bằng:

1. OpenCLIP ViT-B/32 + linear head đã train;
2. NPR-ResNet18;
3. AIDE forensic dual-ResNet50.

Mặc định notebook dùng toàn bộ validation, cân bằng real/fake riêng trong từng generator và xuất metric tổng, metric theo generator, confusion matrix và prediction từng ảnh.

## Kaggle Input

Tiny-GenImage đã có sẵn trong Kaggle Input. Dataset checkpoint chỉ cần chứa:

```text
checkpoints/clip_linear_head.pt
checkpoints/npr_resnet18_from_scratch.pt
checkpoints/aide/model.pt
```

Notebook đã nhúng sẵn runtime của cả ba model nên không cần upload file `.py`. Nó tự tìm checkpoint theo đúng tên file bên dưới `/kaggle/input`.

In [1]:
%pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import importlib.util
from pathlib import Path

import pandas as pd
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Kaggle inputs:')
for path in sorted(Path('/kaggle/input').iterdir()):
    print(' -', path)
assert torch.cuda.is_available(), 'Hãy bật GPU accelerator trước khi chạy.'

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Kaggle inputs:
 - /kaggle/input/datasets


## Cấu hình

Giữ `dataset_root=None` để tự dò Tiny-GenImage. Giữ `max_per_class_per_generator=None` để test toàn bộ validation. Đặt giá trị nhỏ như `10` để smoke test trước.

In [3]:
CONFIG = {
    'input_root': '/kaggle/input',
    'dataset_root': None,  # Ví dụ: '/kaggle/input/tiny-genimage/Tiny-GenImage'
    'output_root': '/kaggle/working/tiny_genimage_three_models',
    'validation_split': 'validation',  # tự nhận val/valid/validation/test

    # None = tự tìm checkpoint theo tên file trong /kaggle/input
    'clip_checkpoint': None,
    'npr_checkpoint': None,
    'aide_checkpoint': None,

    # None = tự tải OpenCLIP ViT-B-32 pretrained='openai'.
    # Nếu Internet OFF, upload backbone và điền đường dẫn file tại đây.
    'clip_backbone_path': None,

    'selection_seed': 42,
    'balance_per_generator': True,
    'max_per_class_per_generator': None,  # None = full; smoke test có thể đặt 10
    'clip_batch_size': 32,
    'npr_batch_size': 32,
    'resume': True,
}
CONFIG

{'input_root': '/kaggle/input',
 'dataset_root': None,
 'output_root': '/kaggle/working/tiny_genimage_three_models',
 'validation_split': 'validation',
 'clip_checkpoint': None,
 'npr_checkpoint': None,
 'aide_checkpoint': None,
 'clip_backbone_path': None,
 'selection_seed': 42,
 'balance_per_generator': True,
 'max_per_class_per_generator': None,
 'clip_batch_size': 32,
 'npr_batch_size': 32,
 'resume': True}

In [4]:
import base64
import gzip

runtime_dir = Path('/kaggle/working/three_detector_runtime')
runtime_dir.mkdir(parents=True, exist_ok=True)
embedded_files = {
    'test_three_models_commfor_unseen_kaggle.py': 'H4sIAAAAAAAC/+09a5PbxpHf+SsQVN0FkLnQLmXpfEyYiqxXVJFllyynrophQVhyyIWXBCgA1O56j//9unveD3C5svK4VFyJFhzM9PT09HT39DR64jj+c7FarVnU7Kqu3LBoWTdRdwF/y2u2iHZVy1h1siw25fomelZvNi/pfcPYyaZesHXEPhXrXdGVdZUNBu+hIRTvAF7ZRmXVsQrfFGtoXFYLtmXwT9VF9TL6rl4U6yyKnkZV3bHzur6Mys22brp28OEDIJNrwB8+RMum3kC3AHRZrtkw2tYAHHrooqKLPnx4eEmDeFhW21334cMwKqpFdNWUHWsH0Hm0YhVrig4GhM3bqKuNRld1c1lWqw8fskEcx4MB9ZXny123a1ieC7QAJCBK+LSDgShbzeXTRdFerMtz+bOs5dPPbV3J503RXcjnBlCsN7yzLZRDY9nTD1iNj/hmC5jJ8qfVjeq52m22N1HRRtVWFm0BIhTA/7YLWdbVzfzC+pFVFTWr3NJsuavmfLawwkuOwQ+v38juX2+KFZCe/rwEMvIK7eWaFU2VbVjXlPNWVk4GEfxXzOe7ppjf5O28btiQl32CqVixfNuwedlCf+bL82JdVHO2yEMt53W13FELIGRTXvPS5ZlZJwgVyoALrJJ6nhe7uSxKBbmRFJ+ouRxHB/PUwprYtF6VjBaAGnHD2op1Z98MxdPjU9Hi42KTFTtgOQkSCgaDgaJj9ub7p8/z9+9+evvs6fsXz/PX3z199eLHaBK9b3YM1tTTd69evM9fvXj74t3T99+/wzecuvHLsmHL9U1OoEbxMFD6SJW++el/Thbsk/W7nV9UbL2WZc+BShoO/VLtXy9YvWqKzV/OvBLVgnqssEk6GCiE85dPv3v95jUN6TaI+FiV9IwhUEEOZ8yfg8Ny3onhQenzp2/enLxwBuqVG0Me61+B4YfeClKM5SO82KvJfPfi6Zv8x+9/eveMqJLEb56+/v5tPIzid09f//gCH0pqxTp8Xi4vPuLfeT2vYyDtYMGWUQsLKUfpkqD4GJPUAMFY3KzrYjFGYZFGJ3+I3tYVG/OlARWybdGAAM42l4uySfiPdoKMNozYddl2eX1JP1NqclV2F7xdDaI7ia8AC1bN6wVIpUm865Yn38QpSosLkD1r0Q/+h3hlC5BRiUBoKKpg+xblatHOy3Lysli3UFaSWpiM1NguitHjJzkKa2N0NJ62a3g3i3LF2g6oJ0RvxhslYcyb8zCiqO7mF7sKtA+s+Y41ybrYnC+KsaiZNaxYJGeno6+jBxH+SYfReRynGoLGJdttF6BkEoKXCskDSqSS7y/YNX9K5EDZNQiYeZe3oFlYvijnXWLO4BDl2ZI1DUjES3YzxtFH/0tzCgPHP0QTbDeFV0MhzN8DjetmxnEslzYQ0o0gwCroE0StniCEktJbuz4QRuKkBq3xBTzE26nVbEZ12Rq6D3UG+uY7MhTSw0AzgzLpYZAa/5gks0HU+PhB+G1nR3f7eR1+VleEpdlLz7SK8cQz0fqo0R+Fx2FIAkBrLLWmKFsWvb/ZshdNUzfJMv4JRMEWdSIYZvMLNr8ks07LsFuwf1S36T5OJUMX1U0CPIbMAQYjrvUk5nZnBssc17RgXI1aH763CKdhm/oTQ/4trw1I4whtUCYBDsVPC24GQmPTJuneXO/6tVjnixIsDTB8cliyCVmpeVPXnZTaKOiqYsNogQ9JbuUXQAq+4CcRCBxc51iZD2MOs12isGnhbUsUJEFJqNIDijPVT9as1vV5IrtJSSigcCxbLmRTRVrdt6IX/oIZmkRTrwsDD7Nttq6vQJSmnFQNoZbKwpkCDE0AoYTDT6PJJDqz5aogJ68wPbVa8kK7vkUWXmFgdKRfp9FvrM4a3Jg0NMj4r1Wc/QyMCAwaRSfRLSK/j3uGnUZQDPWSCoVx7DA7mndv6+5lvasWnOktdJfxi+stmyNx2TWoAdgloWS/lfP0m2YfnTMgW3Sr53L/O0AF4EW3zpD2Y8DcgX8rB7b/a/Uj62hnR4Ng19t1OS872phFz75/+/L1q0y3trSX7gOngDM02Lj1GgwQjheCbBO0z8vV2FBHoMJmjoZCHhYqTCooNTSgPr4WgKaxfhPPQGOSpiWlyXHP6wa4t1okUjV6C+ngGiIJJUDBe95ptmIdwktVFSKXQExWpwWk2qKY61vhGiWOTWqyMGwonWXosH8PE8G8Aop7ak+8MJZcmg6cpYPFnHKboiqXYHvkXIhZA47VS6wvoFhlY3Pi5JqyQWrcrZaSdnblNKAgZA1DpnkiDGzgzQaWYs49E7lsk83bT3Ga9qKgYAMLS3Ggyrjo4fNI4+Mydl1u8/NifnkORUGq2TUM0vkvFGv7QImZQp1pfAS9A2DLlriALEK0APCHXy3EYP3MFejG5DT/NfKdKTBuVSccltbtsB/yVq/F8l6DoShalxUrmvwCzPFsGyw2pNdQI1Btm3v179Qf8hK5uc9xS5+386bowM7eeu/DOBTlgt2BhFdnKGw80Qm+jk2Y9ood2ww/dCbA5tFxgEN4i72Q7vM1K6qEr2q1k1Q7L+BEYQi1nPFABSaGnUgvQfCB1SbM1WoLHFhJkKnJhZxj4tg2nxpRU+Aj/HdILKR/w+Z1s0j4n6C+UbgaEHl1Ie/IyEexHJMGN98VsIUCo26O7j/+Nob9XFVfVWrvTZvzY3Gh/X9G/wr6UWtQqAVIBbNn/SIe+i9ibaQZxNZthmZPPol1xQxkGGgqgPvu1bfxXUCT8xvQ/LDrhT9F0xQ3genjHdNOu6yzb7HB6+8NKGl6vz4dg71cmtgTRQgpmB1D/gVNx7swmwpIszCKPb1zUX+vzo0eqfXsniRZl213LOFpSBb5g50d3pBJZz+BQU80kxsyA/BeLYnzXble5FI7zwsQZT0G4TCqd529A6J10u22azbdLrLnAPllQyYTvhRmIjlTsdOWdcrpihvDXBQKrY09S2vS6Ch6GMWiYmysQqqOal03eyicb/xd7EA91nVmgDimCTduOX6AjjkuraCkXSypLCpwOTbTQr8F5dJN3NpUalXrGlZs0JVHKFF5mrUXu+USTAVdjbHFBFSSMstbtmZ0UpDjK+BkDfIc2oIN3Ja/MLsJB5ob71U7Ti8OlC1AoBZr3Je39a6ZA89NZ7T14j9xsxLwoe5tEMviEuf0VqkNBUWVGIC0Z12Agf1ORVvB04F0EYoyKZWxNat2GzpUSsQ0DCNyR0zOjGW6LmD/BpCQFqZAp3LTXkWTlNeFbm1hIkY+8ZQYUirnbz09ZQIA2Jp6Np1ROaMpbJVOeW3QXr8nxC1gJh8SAlucT47EzKrpbGa8SZbdZMUWzwXF0DTq9uYA/9OzNzloEnjDt6bdZhOPAlg6VQ3uJgLWJyKoNvegg9NZHynQ5bVe2zgg0rwe7F/+wFmsd2JsiiyVxdN6LJGR4dUmugU34e7Vf4gmx6GALQMo2CQ8B0QvTeLIJeugsSmuc3wluAO1fAgQ3+2WLSjelRI/A3v5je+gb3RiEcQ6X+CVhv0kFy5E1y+mqPv7YyZ3bw1DisCBt3bGd8/UocGoesP++bvfePqZZa8cDebkABLmKN0d7TseyBB0t0mb5uOu7gpuvMG2eA2cUyw71oDS4Yy0j5r6qh1HrjuNSC86n9yaWO1hv4Pj8F5i6d50rInzZwSPNt3UNo5m6GadyVNrJNdd1aAOa2xldUhVjgOsmeP523VYuYU1g2Puwmik6PJk3a1XYu9fCX/Yk9LfYbiytjhWTb3b4qlqe1E0Aquczp0YWMxxX/tiA7YlHyY0tkYdbsEV9Dg6PfSeG16ADGLR17fmahvrO+vnPOrmLvjkBsMDh10rq95NEG6dCvT9LSjfGPe0NfbOdltzU93T1tpbjw9su3vamyaP3dwyhvpItTtH899uKAr72pAFv8g5XYT7ZCl3Eg8NK//hLXHw+PTJYp9tq1WA9HvbLnC2jLTa+ywAvdK/mkRnA08aB+3ZcVhyH7XcXTPoH7/izYVxeexSdwd9eLWfHbXaD3VvrnatKo9e6X7QjDEHxwkA4Rj3gP9bEvxLSgLrqAbP3w3/SYLLNFXCAn6Zix7j0JJfyi3VUtZcikEkXbGekMWG7YfRgrXzSfwMxobRiNKI+uHtq9YMhIFaXVlRhKTnegFA0wABjQPdHo9qmmGcU2LAHuJYNkU3iQGB0FGV5805dGRkNc+6OofCxPalU3TSNQ9VEr4KFXqlXTN9/iN0iYFA6liGDQxWuHXOPUQ1YCrtjXFW6bze0emBeXjlcGvMiYOCYF237Zq1bfTu1bc4WQ4bxmLCc2Hzol0kvByDsNzmAVZYzwjPsmhlILM3nTvCWynrWh5A50BZwYNNp3AjDvR22/cl0tHzuOeAOeBqHBzja3T5iTqZOmcuM+8YVJ7I2Q5hcdBrHhzCZrq+yi+WIJvXazyHiYc060ee/3riJH5bRwe4XAQMwJq3AqVpX/+nl5FEAlFflG1xDhuiLO6RUNsG927YIZCwLhawd+IeYu5JpZ5+J/aDKC2wj7kjOciV+6fdaoWFL4s5xtq4nu1+f7I1qWlIAGLMnr+QQZTUjTh+bhPXHEmhFfpSabEni6bems7cZUj807SWmq39k+ygzMOtm7OvDCoRpDPhlmpNQnKcS3+j46k7GC1XxVZyV0mXuEUSGQ7K/9isj7LfqI+ykaLwpNwQU4ZHFYuioT3N4gYaWI6iRPX8kLsve/RAmuojaqmrzBESNoZAEWvK7lzTv1jDbr7CGKwDow7VDo2BxuEtObsL73XoEMF+v4xphpEiv7Wn77czY8btZaip5HvT+ihmr19yJVrjtWXO/ViXs++XZ1vNukosaBYOkF54Wyz8jmW9wQEq9nG9wYOfw+IDPT7t3RobMpZ9KtnVkcFoovl0fHZqDObIALTvpOC8KlouvIcgejseXCYgp3su3BdoRYgvaGAFKbz/Wt0KjPduBBnXFsv4J5odpTH4x0WG3hiLDqVRs+cvW64qbiWJZYCTZ0/ICsKYoOMzcw4Sc1n4RoSw3gQQ40zSO7u3DUx75s0+DrDaHYFfoofECWw0+xs4ksY61zxGAinpY6F8lBgywuuNw2fSrxhfX2pSubEH7hH0YDBfF2Cevv3h3ZvihjWJGxOOM5mDQi67PEfHxBJ9rHNyYVNsCTo9s8dDNFvXTJeNslMY4KPs1Az53W0x9DRT4FLjeHO9zDhcaMwf7JcEH8/f8K8OQ4Q1eAW6R2B2PbaC77mVaRRY9uA1fqywZdOT0Sz6j2gUiHy9hv6up+NhhP87OYN/ZkEAZ0cBIBgawKK+wi3aywwjZJttvUbPz7UgZM5JMDHoMsSP+tgkxlArYA8RnLIBG4zlVhttMeF/u63XCfYM/eDxsKbBeOZ24FmDyTUIv902jR4YU2Jy0DvWvsUArGOYqNptcmrIWjoMQZ65B69UW2QUxbXOWxlVRaE9PCgsuWLl6qJrJ/TBRrg6LY4zaAT4P4PH0SJ5NIyefD2MLkHX4QcJSLNHeMbclECqM9z5LOg7HHg8L4vW3KD64DfF9bau17yD1/i1Tdnd9OGeLee84huKrkvclzCl+ZIV6J9pLXKmv351yIAts8tE0j25TmW8yWLeic8AEyQNTWQPWF5Nn5ig8ixRczZFBaoBm5vTL1Y7fqyZtR9BXp2RQKFqFKBDZ/Q8OlNXGulKxjYAO5aOnCmH/IA3mtdtkvwcfYUiLJWF2xKeStUZIvqzg+jM0oB8sB0NNhFfRUYLjNKZ8FckFB+NtLR9+vr5i+fP3v+AAYw/0i6tPkryXsHGCPasitZAoUcjyY6y5OzJMFo1oOLzczDM1ep6orZsZvPR4yf3WHVG/yKewihx6nKkRDX+w6lhICmqGSVOXQN1Udcoceo2bFW2IO5EoEsSA5uCuDSZ1RlMCguZNS22qjp3Dc9r0MTo96IPGviMFoIZXDhaupfFCr/i5VtZCllWUFAhkGcEGM8sxjJLx2yK9hLWNnmcWlo6Q7l85BI6L7hPwcDHJKITxIKhMahD3bl8IPS1Ny0PqAMLCKyie4JICMmvorPU2eS0l2hjJQ6t/jDhiKbRf0buu99PsH+w4mlJJT5A5bHFH/ZrTkbzfdbuNkl6F/cg9jnBjuW3fYDf/JJAtAdZ5wBAjo0DkRfeH6TtKyT4Qt2pDoR4msIsPXhAE3IE+8wCqLin6L6e4eZnQNeQv88sHjqbgNnYWHXr3aZqyXjZVct67ZxxcWt2V7Ufd4z9wpLT1NbSLoMqpW0IJz2UTIPJ6Gvzbd3Cj6HJs1uU1iQDBG7orELrKUHTEK0Cv8te+TCv2XJZzkvGVzZVBAEV/VF180dV2IvRul4BY65AVO9I0nLSQmkipNQ5fg+kO0pxEYJJriMFkVO4L05r5s8SLfLLDN9HYyHpvX0gjQy5xqb4OMtwI5sAWc/uR1YxmbCsF+VmggBAPT5KUymYjKXHO7IjkTQ5pJgwvpPhdkndEK3scZpr2ACC7L2ZnKXWKMXCVCM8MSZUo24U8q9wUACWc0sFrfBNwlGyBAQ/PHYbcPQsvKc27OkpKBinCLYsXlmgCDZRs4CjSyGzmPPPuzQzZiTRc14DGdtF3IdiLBR3TfDlovvSqycExd4PDUJRgfb5C4mUxLVEhp5t4hwE8Q3Veck/kHHOfop1uaqAGUFmNWLj4FSA/QFUKtqJfRhl2LasqNT8Sgl/mn39DWzI4c/jJ/Tn9MlM2qNydBn9TBUTPqKVZjJdtwgAHo3+GyGORl/zP48/A7CeBRXnANtKHAmt0m7hM49adfzhHpyjNzIcO9jgdeVqV+/aBPSFc/pHeyj1mWXMY/QoLJUvMfeUD4+PRN6TWHg+EgdnWFFuqwtY/nc2O5uFOhvd0WoU7uyuZo/MZntrk/KnH17evS+5xwbiYrt0t9ePTm3F/VhvqkcHNtUAKeOiFNTwx10JA8lRPeHqJjOlzzjBQ7f2s3bC0CX/FrKV+98NRhvKTDM5JeWgL64BnKLaWOTUcTwSj08tj8Qw+oU1NacaVCgXu2JteHIOeShOfRfFfylrZ6Sp+cinpudrcJwS8hhQVLNY4yXwTtWWc+70geF8eT6RHOi85IEtm7IiB0FwBoINiutjG9Aoy2VJMZ0wrh+BwxhJ40Q7ZEanX38zlOlIoPjVizc/JfxR1MB3YIGkvZaysPWO5kaSOFILKjIkijsFPNxgguBJjeQBKHOchsV1sOGZ2ZCkznEdjrwOR8d1+MhsuBRcheqBj/Yrif1XAhtRMEJ18XV2GlyqegITCVGu2W3DcNOBUW3nRUvhOf5ZBGEsE0E09XmBOry7ERIUHTEL/BwbFMQwuqBlrH1fNiw+edxEgEHRORZ0mRrFMmVNX1SKjKaj6BB+uOVHh2FE9lJG2Yxvg0cJezesKsZonk8sp+EIhUfPPfX4UEVF/sMLfaFAck0ypXWMMreNmBLQSDIGEOEbDdAngC46NyiG8qSM+WyZKswLeuGEBg74o8i0VpOygHVvsESOX+Qm1gH82ArlGjqfoYnMIbxUfTVsli7Yp3Ku9sL8l0yyBgtAu+KG+pN1wAaMW5AgC86APFjGxEN+wUpf3+HRD2EugjCIFlFOWYuAhHMMPproWtm8YXhqy9ckbkh0djXNgfFfyvcn3548GsUER+AzcfAbivFN+B+D7PyracweKEQrfhtuubMfn41QNIrzPJ49R+9gYWo0QWE5Ftt8Xc8p9GwSz7c7QEtoz7yu1jemTqOP0+ns0cha1J/iaQhGEn63nnKVOe8MrSuQJohdnfBRpuaw5NcXd4do8Pg+bkrDw4Z1hfj4mHv1yLOn1cR6114kTu4w49xSwAl98GpH87nbP9EwNcajGtB5othc7jogd9uJOjnZ+Pw5w+eAfxvdzqyiuCWzJvrp4/luUbjpuuTZBdKOcwvpYnFUzJFP+1qox4fqEdZ0w/fO6I65ZGyLzzSZqA42W9JcZ+zkbGSDXderkpwwOM+JBGcGW0k5VJrkrJcdajTeWvoj+Y4ftvRck2bAqmDlUJ7IxI5xSET8N4lbqURSS9ug8wVDQyWvDG1UDkSDezqOs+KU+pw5aRsQKIutnl20NOqCfzJMTmCaTRJH9UIf8og4dx7ToYPcKfJVIKaiW9+8/iHiaEUcLfM7dx6qMAkEFlBQrZbLPrqaLkIkcgZLAwMQVeX8cD+jIAf/4RFFfFol1xZqK0O82zzMF/bAfcYsrkJ0IxsOrRReqznY/Wva7aroBc6FuLAwfqH4VJRrXHtmEINRhW223Y0I4LMUox+p/HkBendoVjw2/Ecp1h71yTPnTqxz67+FPuLC7XiFFFJEElcOK6yKlBpHtJVKh13ihrxkiivs2DGjJlABiJag6TXimRazs8eu08OEzNCH9gx4AOv313tf821M0l/lLcZLr7H3Hu9V0PdkwJtZJt+/dfJn6mSlConRPCX8r6gJrcRB/xRKEATSiZRIX1j/qSX3T6z+/rUUH6aM+ntovh4l14rgEuFM82JObPUSdOt9KZXoaLO/g0YUwfNGwJFYf7TqDigHuRKRHtojJSkS/7olSembcCu8zYqWfvDFKMU3lMt4IWB+lJawyRk9fmx4urjfh683NSP02RaXrzxDVLZlzWYHQgY22nRGZp1+uMfa8hSFfzgm+SYxu/on0UoCYevs3+CAkLpCenO9dJzKOtUqywQnVQl+4OLoFVcBiWxxueQe5XJ2dUyvgB2EPvuRnUqXoZoz48grqP08MStn+P+hwKXkeCLyVdx4kCxJ5lmd9Dpjb/KuoWAFajQVHx7PUBDw9cM5ViUKvaEtsq7vuTi9pjzrnmrMTzsFGOHKhMXP83jJXjr8qHIL/4e/3VYkuTRvWkg43kMBcshzFbUTzq1Ar+ITW9s0N3JA4uEmdySbJ5s0otRK1bjb8MwIvFIiiQVYpyJeyqlNX2J7tc8CtbtK1Osqs3i5FcXLrVUsay+t2p2s3Vm15V0Vyt9sX17hEM9Cy7vvQsHouQnjEDCR+gHvulBgzKsvvFmE3Rl3e09OxSHgouR3W0xOLchLfnL1OZDPjoCsLu0wPPbWNR6/Ev7yTAGWt4V8PkRxdYimg3mViAW2PrcaehefaH4JX4lyCJi7QgHWdCr5W3I07iIlF0u+nc3sVJ/imh3hlU/0l/vqCErK8dYxHyMhAIFQjfHJTEjsiaRKBqyp+SzFIC11YTLx/EXHtBAfNmDyZbzrZxLd2qczOc++/OCBK7gNgCJ9u/6w2xgb/1pGg9rnoivxRbfsWey5dOIF3wg8Nl3HvL6oG/ExK8w0mLbJFGnIkwxN6R8j1QSRQeeJgGkvVxUwENdgzlcZNn7BJB5+Ao8QRQefm/ziCyS+iPn85JxQfnaQfAs7vPxwth2fITg0x1e0d6wh3ZG88shJumATuKeRSjVwiM/8VpiywElIIJYL0jHAceLmrrKidUmW+iW7aROf3irrmmFcCbBiMYZ4T84gsSB/nh3FxgrwYW41Bva3YVXNgvzhGF4zsP8bMpropYfLDMKEqh/FX06TPubS8dDTgNFz0JrpsU56TIt+uyCg0QMq+aCyHRof3W6KeVNb6fEOcs2DB7fLmNoYS/KWk2WvtLi3Wqe8xizDCENYV/yyISqjSyQ4WffBfsTUeJ3YU3bvHuKruoF9jka13wD1hxOYX+i5rGxL+5x94Q5gsyw7uJeW5oSEncCmaG6krqbCvo2KUOSYmoo/WfnWoSHlWYe/Rrk3CivplCiz2NecQCV27JrSOLPvQezJp3zwQihtgwG8YrfuWq+5tQyMKzdA0FnpQszlZWTgMOuJqxQfOrlI6L5IbmC28fDQ5QAUj3ggeb/z3k+s71Rws+Q7r4PJ8d0+vFwtboaecD5mIMv3V6xq3xTnDzHxya6C/bp0cbYnLz5ZusLJ0gyN8fTMreRkXh5HXz8y3waSLI+js9PTU4siTk7UcfT49NSuE8jISXBcaW4kIR1Ho1OfsNITD68fjZx5Db5VGT8FqxvJpTj3qjvW6H1qpAbCCDdRSWQ3xws2MyRUcjB7NQdSbbN7tuCeJ+CwHeYbO7LRvZ1aBvwc05Lc0YcR2xe8UsdcuDOZo0Ftg0iQemnTjQoiX5Yhdr3aUrIJ2CBJK5TpfaDl+1iZrFCR3INktSYOdtauc+jAN5ORSSBHJ1+nHE88RrrnZiM719jQTjV/IH+VTFtEsJzsReISIvldAjrGvH0hfsRzd87er6ixIkAg9Ss/qHaA9WQTdq7IofuyJJ5uxt2/4MZBXiGgEopcFK2X1IOfeqvx3sqnfabvP+jPDpMtdnhrCt7jm6QZ3gGXHkDFaS4vawAx/HFHCaeCqd8c/uxJqpXvWvyiw7eX7bMz4KGlbWbSTVW3sdA06mq0IWWG9ZOr0cu9dWRN99DJ1C/EUaF0y3ipHeyVxNV4hoYUFzw6VwcFjStn7SI1FJxW2lPOYN38eB45KQeaaHsr7+1K9D1I/jUy3LFEX9v33cYnwMqb+FTMOx7KqEMq/jPhxzwH5TDvjA4S7Sw2z8X55y2Htf+dzjsm/NteHhvzQigdvWp9xSe5AOzfgD0ys+b1QD1vPmUOfbDWYN9clPHAzMtDUhU2A7b55wfnjSN5IaofquzmjBqGctbgJWhWuTkGgzMdL48ZV6UIYMgt18Jwvz9y44V7L4dSERj+SFXo2K8aqGPE3nOcjqlkDtO7Yso/8fPHpKMCvEGZA5HIuxb2TAZeu1jsB54xcX6Ty7PyW3k9Zov2WeANLmrTW6sYlMSc4la5xseBg0mVrNGxZ7yNoVFB5wDVV+Wa4Ppu5lPf6jkJB53mvyLloBPhojqku+ps1Sxu2KKj95x9BHMxcFuFbB9Qq+oEMR0e0r662l3XWxgEplxZoutAPS5X3wFb4D00kUyCCqaBMWEmvMy528S/JMRHQLGPg7lTU5oAzhwGNHy44+M6/ewOQytrqqk0sznf+DTJXnJOk97jH+vkx7K6pb2sfLohzJQ3OeTdlQMPjk3YGni2yD0G3nq1CDQwk5ItmBl6yUNYDokXmistYOwReKIG4U89F6p7IO+cWh08sk8Pwna/TfJAexVcyAjV57I7aJxTqz5nMLYx7+HRbKDDfDnPTQO+sZm4+AXfU4IHhzsV24ROBYypRizE0cZhFBx325fsX40u7Gs3qXvUQY49rruB3u29586RDex9y5aHDpknCD1h2Z95ovLgQTjt/1J6VXO6szYWd2qH63qWvZxG6ZmdeZsdR1dSSl5vD0GX5/lJ2EOjkD1yf++s74gmJFQOsJWLtRk0rmfo7jmX3uyeyRavbZve9tXibsP0QIScpTpYpyfTuK5qxuto28E8zw8E46jWZvxOsHUwlKdoVsw4YiA8YYIDjpOhc76DVW+9NBky3uHWOADwGG+ojwAcFvliTLE/5pAj1uzinm2In+aeVVeeTsUylgM5eEYTqCQP2WZ2BjrR6eD/ANartORKigAA',
    'test_three_models_tiny_genimage_kaggle.py': 'H4sIAAAAAAAC/7VbbXPjOHL+rl/B8BM5oWl7M5u6ckpX5dvxTlyZ8Uxm5i5XpVJxIRKSuaZILl9s6xz993TjHSApyU7i2h2JILrRaHQ/6AZavu/fPJKiJx31fvl0+zXy7r5+O/tG2zvaXf4p8kiZede3H268qoT/qPcjL3dnH2l5uyUb6q2rIqONl1b3VdPFs9mP+7z1/oNsNgU9q8pi5zV9WUKHhvYtbb3unnr0maSdl9GOpl3VYIcu31Iv39YF3dKyo5mXl7Pffuto2yXdfUNpsq0yWrRJWm2366pJ+rKltEwe2DBxvfvtNyYlybKW8W22eZm3XZ56Wd6m1SNtdtEsLzP6nJebyKtpc7ahIBXB8VekIGXKXlCuh7wq+bSrvqv7zhN9oRmm23hkxid91na7wtUHn7r3gXSkpaAQ3/dns3VTbb0kWfdd39AkwamCtmCIsuoY33Y2E22bVH7jH0W+ivsuL2Tr721Vyu8NyFht5VO7a/lANenugUyO8hUe+YtuV8M0Zft1uVOjlv223nmk9cpaNtXAHBrgvzqTbaCuVPD6evtJMmITn81mt5+vP94kN3//cXP3/fbL3Xdv7r348e/1xo88/KT8S13yz9W2Zp9PdFX7+9n3r59ufyTXn26vv98w0pkHfz6sSJ4xHflXXoCPSMRa1Rf+Gp7QYvwwUpSvJWEPSMO+8L5TtECzn326/svNp+TD7bfkx5fk9oMWuyS41MDrQrBuKBNHPl4kbsOZ22B8Jzk8XIqHNXmgxuNl4jacuQ38+3728ebu5tv1jy/fkn+/vfuBSg4E/2zri86rfLMhpXzawHSpfNjm2e9V35R0J1va7PK98f1n/f3RaH/U7R1ZFYrf4x962Kf+oUK7mIVgSBlde0VFMubvVZkIhAjCK9ZZPCZo5zAHtO8AnCsvwLXC+Cnv7pOSQHf/dADxQ8Y5X3vgkdYAcd4y1nJw/EvBNdAMAM/mXgsuQLNAvcQ/JpJ/ztmf5yWAiB/GzaaoVm+QCv/0N5CxoGWgZQi9f5p7l1eWAA3JW+r9CnLfVd2vVV9mN01TNbaUTPF/rVHTHJMBrRHgTxfQK+kzogLAOqA+R/rYtwbRgjvLpiewuFiyTm1NU3hh416MrQlCDl/hokqZ/4EemYRyG3FsBZzVHI9LAbPpAZsHQ/B2PggOF+A/nARAVbxuF9gao2UtgQVvVHLHqEWYPH0Gafm7gH9wPg0FPCgl1Wz2y5fPn7/cAZ9ROxc+UFbNFgDnH5TbM/5z5bVdE3pnf8bPK5M1vgUpnmgTgKnRuiApuMAZopZvNCRug+c2xLxByADLVJV5SopE7ZgB3/6SCXmU1BlMz5mCQak8TkIObPoGrbZnMT//L/nm4/Wdr8gYfBwj+tt/fvisSQwEO0b4WXdV5AKkjpH+F++myDiIHqP6+AliLE1EBpPzIPTwNz3wyo4yuzZmTcpd0FUPtHTYYSij2gOB3xKvJVZn+Xrdt+Bv8DIMhwN9/wAvXj/SezHS+7GR3k+N9N43Dd6wJWGrawjxkoKsaJFkedOCGxd5h1+vGCIzM83ytFvkZRexpuWVifuqP4I+fARDMV72pggv6rUTCCzS+7zIYtMnl1cea1QkqBXWgloxhobwlY1tIj5nJ6ViselwBOTjyMF47IV6GtpWxSNN1FiBcmmtJMBN+kcPOwCYlfJtfOH9t3cHuwPXCAuDixyCQxjTitziDe0CxSLyjO/h2BYKGGFJ4Z1zxtbsZWdbA+5yq26hsxnyxVKvzRXEKamIo3pokyJ/oIklUYAbiFSODgC5GrWOVlVVWNYkowcuMCY6A/3XozxhIVtD14a0v5KipUdBlg2sIVbQonPeg+GP+KZsduJDJjROS4cNrspPnhIXJTQdtYO8KWmqquPsWZTEnqWu6TOQprlsBEWDDRqmOLEaM2WyajlsTlqr8CQDSKuL5XssIoTGEVA4FGqtfTsxZINlFcSMyBAy0RbEfcHWvRHnicXC5tkg1FxoHUXeuwULpHD92Je8NFQoY813fohTsExxKaKttGrQxwsQZNH1kHsbsIgRzmKpXF17a274UasV0TU7WysMnRpAfODDQc6CO8OjJdwNYW6pONLnlNad9+U7U60zUgVBU9lTDS2wBrisbb8NJn2ajTTmeraYYhY2FiN/Wwauy5jUNS2zIGA9Ij3J0MouhOJnr4rV/V/YYQHzHPvEIfZuniEAxVMTNcVzmNf5C8kjnobuWehg87P6Ch2cv2DmGGEiujfC+NCwlxjTneCB7uYF2a4y4sH6bSFjPsNPCOQjlpywh8tlXJOma8MIvVK2haGFSpwrEMIrAQ5oEUmOU2sDWCoW3e/EDq53H2nzIv1SHQ3LNxzV8AGezLENhDW1/XqdP5t7qHuOYa/1LqcFJxXyrnowksSwJHbQxJcv48dAJqyx9lHg4q8gwwKvRFdkj/x0CrIYUIpasysGy7zDljyzl2lB2tbthsAuwJLDInf0OovxgOrXBjaJyDOfljKngHXJWoEOLGZCAfHMyMQGy6fGkuHFEBSsFsvbTGWNhEHTodCrvVzxXEbqq2HTfMNX26i0jWimvUH4sq2A4z699u8q5r6aUsSxLXxCNw/+p433YupC7g+oLDtWAqVNSaB2ZFiVI4HfIQVpMeejqaDFyAg6pLwsVog8FTKgyJNheoirvm3dHRb5MDxI1EZnQIRi5FBxMy5gVo885ZcMYtmadFVg6jmMCThQ1ebPjtUZHiExfvAe/15GW/nJF8HT7STP/CvPiQyuXqRAez+a5sAngBoGFsZssOUoGfb0rzgOq4bwAFlK0nuaJW+kZmviM/jh63O0s5wYP91kh1zY7M3n3qVHIeoVJ6gH+ChTBDbq+yn9kzXZ5sXu1WRt1TeAzNx/TWrlCwfYMKPHCZvH0dPdpZ3yMZHOsqEzfu0wwWE/aBWA0hcF363AQcxdIBDWrrBO94zptu52JyKdnhu3WAlysLLV0xDkxP5X8GjGikC1bpkUkQFMgAe07Lf4RAOxAWl5F4ZdLOO+zCEbDUIzNtg0VV/DaAbNBDlao3pcGjE7KTCNRT4L/q/wAEZxoXuicR/oebk0AxZky7WN4RuSurrX+v8bKXo6ccS79j8qXb0o8ffetm87Fj0T0OCqAmRlE2GZNco5WLwrJ4bkzJFo/oJRH34L9xGj5i34LdxPHQjDHMejG6u/im3w0DUvAzUSDzXZEJonQsUBevJ8hB5kOhBQYWKO276dnI9JqZ4jBoIHeIavVYgwN2YefFsJyrkxHr8VTNoO/GGO0aT3z0P3Eae9c3aiYO92wkiZwZ3K//Li4hWj0OKtah6du2MViml4ki5OmLwcgNnK8QFGlaGHkQAnQwkAXvDClHTBAmfA/QcSqXxTQm7Eqec/ml6a6ZaU+Zq2HcdsQSqZjpG5VPKrnOC6Ien8Mr4YmQ1eCyBC85Qma6p6hGWcly2FrBAY+LItge0DNsXIK+uYANsNZRqUr0FzWber6Rycw04IZY/IwGN5FVFt674Tl1LJlnZNnraBvNkRMQTPpLwaskrIWvBq/cqz0x1BqI88Z/pgWCU53OywcoAUBbvIZkPAxquHirx37/g9TqxEE0IZw4d89xUdWwJxJ97hB4YY3jng6ItmvE/EwDH2BCWKx9BJuprqqZ3YJiOxrQG0G7LErHG1C4xdLWJJG1/Wq2HczwYZjXqH0e6YjmZvjdN8rqCEV5ZgxMMODI1URctYPZXsPj0B++XX4UN+k2vFdOIEqHvn8EMPJajcgMnW1wRR3FVJ2j5qNR60gSE50Pq4n6Bn66Ng4YxSW0W/LZlVqFF8kqY9+PjO0IovNposGXvJdAkxINic2YyanWoGK0vz1o5i+Zv1pcW6ShPSp2YTQeveDFksBcqkTaUKK46a2bt3tl2ufcbA0OYL19EeM7GiAuwcKHrBeyzjLSVlEIbDIwv2Hp3LVrvquDem91Q1gId6kKHmpSDWOEOpRtYMRIS9ySgTMFeY/n8OS54nhoVQXI/ayiQQsH/AORRkbwFIvqhtv92SZidhkjVau8mLL6AT5BDf2CYF/dB+8DMyUzoxppXMiTZ5i8YEXOMFSgf7bcJLxLgO+ffBYZ/c0pyNiJ/v39P0oa5AR7DVrasrZxuKxGTavuha86VzJBdN72E12bEKD8uDWhh3SxLQSMsrqy6NFaTPEOHkWIkHL2xvknnoIAF1APeE9FaHCjwMMU1FhQnhFI0olkIC1dvOusIYLwCmOYgiqWkOlxMcLOMW+aZmMJZtAu7jOeoBYdp78tPP/wr8pBewZ35WbVgV+IFcgXNWpaNCMNwaXO7atlBSx9KiUazi9T647HvTNxHzzLiHGyTinzDN4cGdiE3QM1ltDLsrhK6LEW9b4sbIfL5qcjC7uS/OH/zQPk0EpmxQ4H3lngvDzlFC4ApAFcB7dg3tQ3DMawpAUTDUs8/vIsZOCqunxbD70pnBeCcto/C1hdTjcqG1trQ8kGlbY5PUjWxZDgyFI5bsx5+XhywTlWQs8jjGOrbFHyWcitlYgCraZEVBXya6ZjboW8yMQD/5xgUycQsBOsCPSbCaILawS99yIr7YRXaGKZuHS9CR34Fo1euJm1xgr37Iy805CzVBnfy406yFM4dwj80n8c7n6RnrCFkV9Hv/0zAWs5Ng6IQxuemc0xnzcIJpkdeJ9vlhh7JuDr4neUYPdmAjrEj6sIJ2eUY83qdL75M2/wfi7b/85Mgw/RaNnR0Kaz3s5YGkaWrGzS8+x32Nl66mNQoTZtltjCsQIOzzdwt3cZYCdCBzfSUFK4+G0KjsIXw+lQgmw+nSPiN4r0UeSV5gRZQJp0YXg38CWHFsDKe4QtY7SArDmwSQmZjgdDa9RvSWbsZvmsb3Kn/mJjuDnnIr4CGLzljHexsdJAUgZ8mcaIJAvvfVhqaujFlVmjEP64ggcqWJnMGMRdKX0NsHXgvTwG7Wstw64iUfSfUgTlBEcNbd4zYpgFle1PFFYS8D04JNTMNTqpEyGnu1I+kTbDu0IFFcy8ulHWDZMjSzy5FTGRDg0NW3K+/cfNBe7o47PyJUZJyj0Wx+0Ph131GEnbPaJkk9DsImkwP4OzfVfAinrUtkFb2JcwHTl875rxHsCM9K/jkPozZL39HIY56Fdc5j3rQZQUuM2Gskc9aRH3ae+wqSxKWrUbA0IbumYMUw7ZT47iTUdP+vpyAPSE+egSI4NoGxPM6OWNguWOQlJU0C4LGiw7zKuGJlbr8YbOBLN7w/lDOcwGPv7MOgshJ/8HWKbE7s8BbRDrDYu1EIoDUt2zwVMv58cYqMbvzyFiEP8dgfPLxwNqRzKxuTAbZjNKLOCnJvrM0UzoOwYlnTA93xeWLcTUMn9WGNkSh3ZvfmrIVteEAZiUesbeOIJRI3Yy4jAix8/jML8AZrD8HIHEWxqihOyjTE6EIRI0MKv8roY55i2sADIP4IiR0EQv7B0EkUDaR1L1y9bnCv4LUXHj+iGa2ykf0+sKGwE/uy/zf2uxhPHFZAu3VQYVO/Dc4EisU9Lmb6EIAlFglbr/mFiZbMqyEq6RoCmJKpn3GxxXMAxI7Pl6FbETfRz7yBs+5WIXOqwa5z35BHBEgVqwA7Anu8wkr5nCBNsKt9DqkijrEfVrlHHVNwZ/fiC2m3mfGDm6y4iOHoPRo//rShdGK+0Ot/P10XQl83Wyf3MicbngLAEzPD/hNTM6bjTQFsJOR2ZdnPBpnBasdTcvO0wCplRGvcn3iAqvtSUU3LbzBb47504d5usnMr9tPZQN9mjh6YKRcxLuTAX4bHZnqC8ndybjI0OAo37/cwSBnh1YpzF6vSwAx+Raq9FHWxthBjP4F0f8MBim8o/pINAimHPGQ1wxzI2sDV4cEb5qnfPIZ4PmvhL5Ob3TU3ZJfQPyBDHlbCKPrTVjJybcGuYxkeItoKt39vYvfjW8w31Dr+IhuvJAuKRU/mylo3t74N3cZIgx+LDEVRFucUNpoDiCjYWbyRiHfKD52zTqOLUSnF/NDpOX6vP+I+yMtKzkWQUFTlJnFmrCojxkSNhTGGk+USLsuhelqV6sAuLX4n67riiPaeEPFsWXWKV4H9HQQQvuDlqP6HYOKOtXCQQ1zcViuyyiG73vnOutnV2/6guy6j1u4Tnjy6eEVFLfCRwd3eY2M7IOwKMLh+P3kxE2Q1ef3OPpDQLJbWBqiL4Q/cgPDrjcG1irTTsXIgIwKzBpfTtO9PzZmdUFkgIk3wTNLkEKS79Q4TFf5vrAlxL+315b24EEleIHnBG3umkPG+gwxncKliJTvun6joH+ZSeHU3JNmPzeLg9cx+UEZw0pWaNnQzBVALc3y15SX5xBrr++Hxm20jcYuMWM69TpRij6ZBWlzbXBZT5jJSDiQt4VDhin1fdrD44QSiYanKK2o6bKKlDVcQd8K2H0ychm1SgI4CT5+CN14RsDrhhNXwB+PXd/8DanuVofVHAAA=',
}
for filename, encoded in embedded_files.items():
    target = runtime_dir / filename
    target.write_bytes(gzip.decompress(base64.b64decode(encoded)))
    print('Runtime:', target)

def require_checkpoint(filename, path_hint=None):
    candidates = sorted(p for p in Path(CONFIG['input_root']).rglob(filename) if p.is_file())
    if path_hint:
        hinted = [p for p in candidates if path_hint.lower() in str(p).lower()]
        candidates = hinted
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Cần đúng một checkpoint {filename!r} trong {CONFIG["input_root"]}; '
            f'tìm thấy {len(candidates)}: {candidates}'
        )
    return candidates[0]

CONFIG['clip_checkpoint'] = str(require_checkpoint('clip_linear_head.pt'))
CONFIG['npr_checkpoint'] = str(require_checkpoint('npr_resnet18_from_scratch.pt'))
CONFIG['aide_checkpoint'] = str(require_checkpoint('model.pt', path_hint='aide'))
print('Checkpoints:')
for key in ('clip_checkpoint', 'npr_checkpoint', 'aide_checkpoint'):
    print(f' - {key}: {CONFIG[key]}')

runner_path = runtime_dir / 'test_three_models_tiny_genimage_kaggle.py'
spec = importlib.util.spec_from_file_location('tiny_three_models_runner', runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
print('Runner imported successfully.')

Runtime: /kaggle/working/three_detector_runtime/test_three_models_commfor_unseen_kaggle.py
Runtime: /kaggle/working/three_detector_runtime/test_three_models_tiny_genimage_kaggle.py
Checkpoints:
 - clip_checkpoint: /kaggle/input/datasets/hoangha071005/checkpoint-clip-npr-aide/checkpoints/clip_linear_head.pt
 - npr_checkpoint: /kaggle/input/datasets/hoangha071005/checkpoint-clip-npr-aide/checkpoints/npr_resnet18_from_scratch.pt
 - aide_checkpoint: /kaggle/input/datasets/hoangha071005/checkpoint-clip-npr-aide/checkpoints/aide/model.pt
Runner imported successfully.


## Kiểm tra nhanh trước khi chạy full

Nếu muốn kiểm tra đường dẫn và checkpoint trước, đặt `max_per_class_per_generator=10`. Sau khi smoke thành công, đổi về `None`, xóa `/kaggle/working/tiny_genimage_three_models` hoặc Restart Session rồi chạy full.

In [5]:
result = runner.run_evaluation(CONFIG)
print(json.dumps(result['experiment'], ensure_ascii=False, indent=2))

Tiny root: /kaggle/input/datasets/yangsangtai/tiny-genimage
Device: cuda; test samples: 7000
label_name  fake  real
generator             
ADM          500   500
BigGAN       500   500
GLIDE        500   500
Midjourney   500   500
SD15         500   500
VQDM         500   500
Wukong       500   500


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP linear probe:   0%|          | 0/7000 [00:00<?, ?it/s]

NPR-ResNet18:   0%|          | 0/7000 [00:00<?, ?it/s]

AIDE forensic ResNet50:   0%|          | 0/7000 [00:00<?, ?it/s]

                 model  overall_balanced_accuracy  macro_generator_balanced_accuracy  macro_generator_roc_auc  worst_generator_balanced_accuracy
     clip_linear_probe                   0.863143                           0.863143                 0.935214                              0.763
          npr_resnet18                   0.966000                           0.966000                 0.995302                              0.950
aide_forensic_resnet50                   0.924714                           0.924714                 0.975828                              0.799
{
  "dataset": "Tiny-GenImage",
  "split": "validation",
  "manifest_samples": 7000,
  "manifest_real": 3500,
  "manifest_fake": 3500,
  "generators": [
    "ADM",
    "BigGAN",
    "GLIDE",
    "Midjourney",
    "SD15",
    "VQDM",
    "Wukong"
  ],
  "manifest_sha256": "4b8d2dabf02ccf95ee72011f7fc8ec84e49f9d4d1523e2d57c977d60e398bfd6",
  "checkpoints": {
    "clip_linear_probe": {
      "path": "/kaggle/input/datas

## So sánh tổng quan ba detector

In [6]:
output_root = Path(CONFIG['output_root'])
summary = pd.read_csv(output_root / 'metrics/all_models_summary.csv')
display(summary[[
    'model', 'overall_accuracy', 'overall_balanced_accuracy',
    'overall_real_recall', 'overall_fake_recall', 'overall_fake_precision',
    'overall_fake_f1', 'overall_roc_auc',
    'macro_generator_balanced_accuracy', 'worst_generator_balanced_accuracy'
]])

,model,overall_accuracy,overall_balanced_accuracy,overall_real_recall,overall_fake_recall,overall_fake_precision,overall_fake_f1,overall_roc_auc,macro_generator_balanced_accuracy,worst_generator_balanced_accuracy
0,clip_linear_probe,0.863143,0.863143,0.867143,0.859143,0.866071,0.862593,0.934056,0.863143,0.763
1,npr_resnet18,0.966000,0.966000,0.960000,0.972000,0.960474,0.966203,0.995252,0.966000,0.950
2,aide_forensic_resnet50,0.924714,0.924714,0.981429,0.868000,0.979053,0.920188,0.975470,0.924714,0.799


## Metric và confusion matrix theo generator

In [7]:
generator_metrics = pd.read_csv(output_root / 'metrics/all_models_generator_metrics.csv')
display(generator_metrics[[
    'model', 'generator', 'num_real', 'num_fake', 'tn', 'fp', 'fn', 'tp',
    'balanced_accuracy', 'real_recall', 'fake_recall', 'fake_precision',
    'fake_f1', 'roc_auc', 'average_precision', 'confusion_matrix'
]].sort_values(['generator', 'model']))

,model,generator,num_real,num_fake,tn,fp,fn,tp,balanced_accuracy,real_recall,fake_recall,fake_precision,fake_f1,roc_auc,average_precision,confusion_matrix
14,aide_forensic_resnet50,ADM,500,500,496,4,40,460,0.956,0.992,0.920,0.991379,0.954357,0.986454,0.989467,"[[496, 4], [40, 460]]"
0,clip_linear_probe,ADM,500,500,425,75,74,426,0.851,0.850,0.852,0.850299,0.851149,0.927120,0.923233,"[[425, 75], [74, 426]]"
7,npr_resnet18,ADM,500,500,480,20,30,470,0.950,0.960,0.940,0.959184,0.949495,0.990712,0.988040,"[[480, 20], [30, 470]]"
15,aide_forensic_resnet50,BigGAN,500,500,486,14,42,458,0.944,0.972,0.916,0.970339,0.942387,0.985458,0.983133,"[[486, 14], [42, 458]]"
1,clip_linear_probe,BigGAN,500,500,436,64,9,491,0.927,0.872,0.982,0.884685,0.930806,0.982658,0.980237,"[[436, 64], [9, 491]]"
8,npr_resnet18,BigGAN,500,500,474,26,3,497,0.971,0.948,0.994,0.950287,0.971652,0.996524,0.994749,"[[474, 26], [3, 497]]"
16,aide_forensic_resnet50,GLIDE,500,500,494,6,8,492,0.986,0.988,0.984,0.987952,0.985972,0.995866,0.984057,"[[494, 6], [8, 492]]"
2,clip_linear_probe,GLIDE,500,500,434,66,0,500,0.934,0.868,1.000,0.883392,0.938086,0.993642,0.992749,"[[434, 66], [0, 500]]"
9,npr_resnet18,GLIDE,500,500,479,21,0,500,0.979,0.958,1.000,0.959693,0.979432,0.999808,0.999813,"[[479, 21], [0, 500]]"
17,aide_forensic_resnet50,Midjourney,500,500,489,11,190,310,0.799,0.978,0.620,0.965732,0.755177,0.915776,0.930284,"[[489, 11], [190, 310]]"


In [8]:
confusion_comparison = generator_metrics.pivot(
    index='generator', columns='model', values='confusion_matrix'
).reset_index()
display(confusion_comparison)

model,generator,aide_forensic_resnet50,clip_linear_probe,npr_resnet18
0,ADM,"[[496, 4], [40, 460]]","[[425, 75], [74, 426]]","[[480, 20], [30, 470]]"
1,BigGAN,"[[486, 14], [42, 458]]","[[436, 64], [9, 491]]","[[474, 26], [3, 497]]"
2,GLIDE,"[[494, 6], [8, 492]]","[[434, 66], [0, 500]]","[[479, 21], [0, 500]]"
3,Midjourney,"[[489, 11], [190, 310]]","[[436, 64], [59, 441]]","[[489, 11], [23, 477]]"
4,SD15,"[[490, 10], [34, 466]]","[[420, 80], [59, 441]]","[[477, 23], [7, 493]]"
5,VQDM,"[[494, 6], [77, 423]]","[[444, 56], [181, 319]]","[[482, 18], [8, 492]]"
6,Wukong,"[[486, 14], [71, 429]]","[[440, 60], [111, 389]]","[[479, 21], [27, 473]]"


## Đóng gói output

ZIP chứa manifest, predictions, checkpoint hashes, JSON và toàn bộ metric CSV.

In [9]:
import shutil
zip_path = shutil.make_archive(
    '/kaggle/working/tiny_genimage_three_models_results',
    'zip',
    root_dir=output_root,
)
print('Download:', zip_path)

Download: /kaggle/working/tiny_genimage_three_models_results.zip


In [10]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/tiny_genimage_three_models_results.zip"
))

/kaggle/working/tiny_genimage_three_models_results.zip